In [9]:
import paho.mqtt.client as mqtt
import time

mqtt_server = "192.168.100.21" 
mqtt_port = 1883
mqtt_client_id = "python_phone_communicator"

# 핸드폰으로 메시지를 보내는 토픽
# 이 토픽을 핸드폰 클라이언트 앱에서 구독해야 합니다.
topic_to_phone = "phone/message"

# 핸드폰에서 메시지를 받는 토픽
# 핸드폰 클라이언트 앱에서 이 토픽으로 메시지를 발행해야 합니다.
topic_from_phone = "phone/status"

# --- 콜백 함수 정의 ---

# MQTT 브로커 연결 성공 시 호출
def on_connect(client, userdata, flags, rc):
    """브로커 연결 성공 시 실행되는 함수입니다."""
    if rc == 0:
        print("✔ MQTT 브로커에 성공적으로 연결되었습니다.")
        # 핸드폰 클라이언트가 보낼 메시지를 받기 위해 토픽을 구독합니다.
        client.subscribe(topic_from_phone)
        print(f"✔ '{topic_from_phone}' 토픽을 구독합니다.")
    else:
        print(f"❌ 연결 실패! 반환 코드(rc): {rc}")

# 메시지 수신 시 호출
def on_message(client, userdata, msg):
    """메시지를 수신하면 실행되는 함수입니다."""
    # 핸드폰 클라이언트가 보낸 메시지를 확인합니다.
    if msg.topic == topic_from_phone:
        message = msg.payload.decode("utf-8")
        print(f"\n✔ 핸드폰 클라이언트로부터 메시지 수신:")
        print(f"  - 토픽: {msg.topic}")
        print(f"  - 내용: {message}")
        print("-" * 30)


# --- 메인 프로그램 실행 ---
print("--- 핸드폰 MQTT 클라이언트 통신 프로그램 ---")

try:
    # MQTT 클라이언트 생성
    client = mqtt.Client(client_id=mqtt_client_id)
    client.on_connect = on_connect
    client.on_message = on_message

    # 브로커에 연결
    print(f"MQTT 브로커 '{mqtt_server}:{mqtt_port}'에 연결 시도 중...")
    client.connect(mqtt_server, mqtt_port, 600)

    # 논블로킹 방식으로 메시지 수신 대기
    # 이 함수는 백그라운드에서 메시지를 계속 확인합니다.
    client.loop_start()

    # 연결이 완료될 때까지 잠시 대기
    while not client.is_connected():
        time.sleep(1)

    # 핸드폰 클라이언트에게 보낼 메시지
    greeting_message = "안녕하세요! 파이썬 클라이언트입니다."

    print(f"\n✔ 핸드폰 클라이언트에게 메시지 전송:")
    print(f"  - 토픽: '{topic_to_phone}'")
    print(f"  - 내용: '{greeting_message}'")

    # 메시지 발행
    client.publish(topic_to_phone, greeting_message)

    # 핸드폰 클라이언트로부터 메시지를 받을 수 있도록 60초 대기
    # 메시지를 받으면 on_message 함수가 호출됩니다.
    print("\n핸드폰 클라이언트로부터 메시지를 기다리는 중... (60초)\n")
    print("핸드폰 MQTT 앱에서 'phone/status' 토픽으로 메시지를 보내보세요.")
    time.sleep(60)

except Exception as e:
    print(f"❌ 예외 발생: {e}")

finally:
    # 연결 종료
    if client.is_connected():
        client.loop_stop()
        client.disconnect()
    print("\n연결 종료.")



--- 핸드폰 MQTT 클라이언트 통신 프로그램 ---
MQTT 브로커 '192.168.100.21:1883'에 연결 시도 중...


C:\Users\user\AppData\Local\Temp\ipykernel_2896\2566096294.py:46: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client(client_id=mqtt_client_id)



연결 종료.


KeyboardInterrupt: 

In [ ]:
pip install paho-mqtt


In [ ]:
import paho.mqtt.client as mqtt
import threading
import time


# =====================================================
# MQTT 설정
# =====================================================

MQTT_SERVER = "192.168.100.21"
MQTT_PORT = 1883

# Python → 핸드폰
TOPIC_TO_PHONE = "phone/message"

# 핸드폰 → Python
TOPIC_FROM_PHONE = "phone/status"


# =====================================================
# MQTT 연결 성공 시 실행
# =====================================================

def on_connect(client, userdata, flags, rc):

    if rc == 0:

        print()
        print("====================================")
        print("✔ MQTT Broker 연결 성공")
        print("====================================")

        # 핸드폰에서 보내는 메시지 구독
        client.subscribe(
            TOPIC_FROM_PHONE,
            qos=1
        )

        print("구독 Topic :", TOPIC_FROM_PHONE)

    else:

        print("❌ MQTT 연결 실패")
        print("rc =", rc)


# =====================================================
# 핸드폰 → Python 메시지 수신
# =====================================================

def on_message(client, userdata, msg):

    try:

        message = msg.payload.decode("utf-8")

        print()
        print("------------------------------------")
        print("📱 핸드폰 → Python")
        print("Topic   :", msg.topic)
        print("Message :", message)
        print("------------------------------------")

    except Exception as e:

        print("메시지 처리 오류 :", e)


# =====================================================
# MQTT 연결 종료
# =====================================================

def on_disconnect(client, userdata, rc):

    print()
    print("MQTT 연결 종료")
    print("rc =", rc)


# =====================================================
# 키보드 입력
# Python → 핸드폰
# =====================================================

def keyboard_send():

    while True:

        try:

            message = input()

            # exit 입력 시 프로그램 종료
            if message.lower() == "exit":

                print("프로그램 종료 요청")

                break

            # 핸드폰으로 메시지 전송
            result = client.publish(
                TOPIC_TO_PHONE,
                message,
                qos=1
            )

            # Broker에 publish 완료될 때까지 대기
            result.wait_for_publish()

            print()
            print("------------------------------------")
            print("💻 Python → 핸드폰")
            print("Topic   :", TOPIC_TO_PHONE)
            print("Message :", message)
            print("------------------------------------")

            print()
            print("메시지 입력 : ", end="")

        except Exception as e:

            print("전송 오류 :", e)


# =====================================================
# MQTT Client 생성
# =====================================================

client = mqtt.Client(
    client_id="python_phone_communicator"
)

client.on_connect = on_connect
client.on_message = on_message
client.on_disconnect = on_disconnect


# =====================================================
# 메인 프로그램
# =====================================================

try:

    print()
    print("====================================")
    print("Python ↔ 핸드폰 MQTT 통신 프로그램")
    print("====================================")

    print()
    print("MQTT Broker 연결 중...")
    print("IP   :", MQTT_SERVER)
    print("PORT :", MQTT_PORT)

    # MQTT Broker 연결
    client.connect(
        MQTT_SERVER,
        MQTT_PORT,
        keepalive=30
    )

    # MQTT 통신 시작
    client.loop_start()


    # -------------------------------------------------
    # 연결 완료 대기
    # -------------------------------------------------

    while not client.is_connected():

        time.sleep(0.1)


    print()
    print("====================================")
    print("MQTT 통신 준비 완료")
    print("====================================")

    print()
    print("Python에서 메시지를 입력하면")
    print("핸드폰으로 전송됩니다.")

    print()
    print("핸드폰에서 메시지를 보내면")
    print("Python 화면에 바로 표시됩니다.")

    print()
    print("종료하려면 exit 입력")
    print()

    print("메시지 입력 : ", end="")


    # -------------------------------------------------
    # 키보드 입력 스레드
    # -------------------------------------------------

    send_thread = threading.Thread(
        target=keyboard_send
    )

    send_thread.start()


    # -------------------------------------------------
    # 프로그램 실행 유지
    # -------------------------------------------------

    while send_thread.is_alive():

        time.sleep(0.1)


except KeyboardInterrupt:

    print()
    print("Ctrl+C 프로그램 종료")


except Exception as e:

    print()
    print("❌ 오류 발생 :", e)


finally:

    print()
    print("MQTT 연결 종료 중...")

    client.loop_stop()

    if client.is_connected():

        client.disconnect()

    print("✔ 프로그램 종료")


Python ↔ 핸드폰 MQTT 통신 프로그램

MQTT Broker 연결 중...
IP   : 192.168.100.21
PORT : 1883

✔ MQTT Broker 연결 성공
구독 Topic : phone/status

MQTT 통신 준비 완료

Python에서 메시지를 입력하면
핸드폰으로 전송됩니다.

핸드폰에서 메시지를 보내면
Python 화면에 바로 표시됩니다.

종료하려면 exit 입력

메시지 입력 : 

C:\Users\user\AppData\Local\Temp\ipykernel_12460\1520909708.py:129: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client(



------------------------------------
💻 Python → 핸드폰
Topic   : phone/message
Message : hello world
------------------------------------

메시지 입력 : 
------------------------------------
💻 Python → 핸드폰
Topic   : phone/message
Message : hihi
------------------------------------

메시지 입력 : 
------------------------------------
💻 Python → 핸드폰
Topic   : phone/message
Message : teeheee
------------------------------------

메시지 입력 : 
------------------------------------
💻 Python → 핸드폰
Topic   : phone/message
Message : hehe
------------------------------------

메시지 입력 : 
------------------------------------
💻 Python → 핸드폰
Topic   : phone/message
Message : hoho
------------------------------------

메시지 입력 : 
------------------------------------
💻 Python → 핸드폰
Topic   : phone/message
Message : alala
------------------------------------

메시지 입력 : 
------------------------------------
💻 Python → 핸드폰
Topic   : phone/message
Message : LAAAAAA LALALALALAAAA
------------------------------------

메시지 입력 :

In [1]:
import sys

print(sys.executable)
print(sys.version)

e:\_se4n\intel7-chat\.venv\Scripts\python.exe
3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]


In [2]:
print("Jupyter 정상 실행")

Jupyter 정상 실행


In [3]:
print("1. import 시작")

import paho.mqtt.client as mqtt

print("2. import 완료")

1. import 시작
2. import 완료


In [5]:
import paho.mqtt.client as mqtt
import time
MQTT_SERVER = "192.168.100.21"
MQTT_PORT = 1883
TOPIC_FROM_PHONE = "phone/status"
# MQTT 연결 성공 시 실행
def on_connect(client, userdata, flags, rc):#MQTT Broker와 연결이 완료되면 Paho MQTT가 자동으로 호출하는 Callback 함수
    
    print( "on_connect 실행됨",  flush=True    )
    print( "rc =",rc, flush=True    )
    if rc == 0:
        print( "✔ MQTT 연결 성공",  flush=True        )
        # phone/status 구독
        result, mid = client.subscribe(  TOPIC_FROM_PHONE        )
        print( "✔ 구독 시작 :", TOPIC_FROM_PHONE,  flush=True        )
        print( "subscribe result =",    result,     flush=True        )
    else:
        print( "❌ 연결 실패",  flush=True        )
# MQTT 메시지가 들어오면 실행
def on_message(client, userdata, msg):
    print("\n메시지가 들어왔습니다.",   flush=True   )
    print("Topic =",   msg.topic,    flush=True  )
    print("원본 Payload =", msg.payload,  flush=True    )
    # bytes → 문자열
    message = msg.payload.decode(   "utf-8"    )
    print(  "문자열 Payload =",   message,   flush=True   )
    # ON
    if message.strip().lower() == "on":
        print( "💡 스위치가 ON 되었습니다.",  flush=True        )
    # OFF
    elif message.strip().lower() == "off":
        print( "⭕ 스위치가 OFF 되었습니다.",  flush=True        )
    # 기타 데이터
    else:
        print( "알 수 없는 값 =",    repr(message),    flush=True        )
# =========================================================
# MQTT Client 생성
# =========================================================
client = mqtt.Client( client_id="python_switch_test")
# =========================================================
# Callback 등록
# =========================================================
client.on_connect = on_connect
client.on_message = on_message
# =========================================================
# Broker 연결
# =========================================================
print(    "MQTT Broker 연결 중...")
client.connect( MQTT_SERVER,    MQTT_PORT,    60)
# MQTT 통신 시작
client.loop_start()
print( "핸드폰 스위치를 눌러보세요.")
print( "ON / OFF 수신 대기 중...")
print(    "60초 후 자동 종료합니다.")
# 60초 동안 MQTT 메시지 수신
try:
    for count in range(60):
        time.sleep(1)
        print( f"\r수신 대기 : {count + 1}/60초",   end="",   flush=True    )
except KeyboardInterrupt:
    print( "\nCtrl+C 입력",   flush=True    )
# MQTT 종료
finally:
    print("\n\nMQTT 연결 종료 중...",  flush=True    )
    client.loop_stop()
    client.disconnect()
    print( "✔ MQTT 연결 종료",    flush=True    )
    print( "✔ 프로그램 종료",    flush=True    )

C:\Users\user\AppData\Local\Temp\ipykernel_2896\234992855.py:39: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client( client_id="python_switch_test")


MQTT Broker 연결 중...


ConnectionRefusedError: [WinError 10061] 대상 컴퓨터에서 연결을 거부했으므로 연결하지 못했습니다

In [6]:
import paho.mqtt.client as mqtt 
import time 
 
# --- MQTT 브로커 설정 --- 
mqtt_server ="192.168.3.45" 
mqtt_port = 1884 
mqtt_client_id = "python_communicator_with_user_input" 
 
# 핸드폰으로 메시지를 보내는 토픽 (사용자 입력 메시지 전송용) 
topic_to_phone = "phone/message" 
 
# 핸드폰에서 메시지를 받는 토픽 (스위치 상태 수신용) 
topic_from_phone = "phone/status" 
 
# --- 콜백 함수 정의 --- 
 
def on_connect(client, userdata, flags, rc): 
    """브로커 연결 성공 시 실행되는 함수입니다.""" 
    if rc == 0: 
        print("✔ MQTT 브로커에 성공적으로 연결되었습니다.") 
        # 핸드폰 클라이언트가 보낸 메시지를 받기 위해 토픽을 구독합니다. 
        client.subscribe(topic_from_phone) 
        print(f"✔ '{topic_from_phone}' 토픽을 구독합니다. 핸드폰 스위치 상태를 여기서 받습니다.") 
    else: 
        print(f"❌ 연결 실패! 반환 코드(rc): {rc}") 
 
def on_message(client, userdata, msg): 
    """메시지를 수신하면 실행되는 함수입니다.""" 
    if msg.topic == topic_from_phone: 
        # 메시지 내용을 문자열로 디코딩 
        message = msg.payload.decode("utf-8") 
 
        # 메시지 내용에 따라 정확히 요청하신 문장 출력 
        if message.lower() == "on": 
            print(f"\n💡 스위치가 on 되었습니다") 
        elif message.lower() == "off": 
            print(f"\n⭕ 스위치가 off 되었습니다") 
        else: 
            print(f"\n✔ 핸드폰 클라이언트로부터 메시지 수신:") 
            print(f"  - 토픽: {msg.topic}") 
            print(f"  - 내용: {message}") 
        print("-" * 30) 
 
 
# --- 메인 프로그램 실행 --- 
print("--- 사용자 입력 & MQTT 통신 프로그램 ---") 
print("프로그램이 실행되면 핸드폰으로 '파이썬에서 값을 입력 하세요' 메시지를 보냅니다.") 
print("이후 터미널에 직접 메시지를 입력하면 핸드폰으로 전송됩니다.") 
print("핸드폰 앱에서 스위치를 ON/OFF하면 터미널에 상태가 표시됩니다.") 
 
try: 
    # MQTT 클라이언트 생성 
    client = mqtt.Client(client_id=mqtt_client_id) 
    client.on_connect = on_connect 
    client.on_message = on_message 
 
    # 브로커에 연결 
    print(f"\nMQTT 브로커 '{mqtt_server}:{mqtt_port}'에 연결 시도 중...") 
    client.connect(mqtt_server, mqtt_port, 600) 
 
    # 논블로킹 방식으로 메시지 수신 대기 시작 
    client.loop_start() 
 
    # 연결이 완료될 때까지 잠시 대기 
    while not client.is_connected(): 
        time.sleep(1) 
 
    # 초기 메시지 전송 
    initial_message = "파이썬에서 값을 입력 하세요" 
    client.publish(topic_to_phone, initial_message) 
    print(f"\n✔ 초기 메시지 전송: '{initial_message}'") 
 
    # 무한 루프를 돌며 사용자로부터 입력 받기 
    while True: 
        user_input = input("핸드폰으로 보낼 메시지를 입력하세요 (종료하려면 'quit' 입력): ") 
 
        if user_input.lower() == 'quit': 
            print("프로그램을 종료합니다.") 
            break 
 
        print(f"\n✔ 핸드폰 클라이언트에게 메시지 전송:") 
        print(f"  - 토픽: '{topic_to_phone}'") 
        print(f"  - 내용: '{user_input}'") 
 
        # 사용자 입력 메시지 발행 
        client.publish(topic_to_phone, user_input) 
        print("-" * 30) 
 
except Exception as e: 
    print(f"❌ 예외 발생: {e}") 
 
finally: 
    # 연결 종료 
    if client.is_connected(): 
        client.loop_stop() 
        client.disconnect() 
    print("\n연결 종료.") 

--- 사용자 입력 & MQTT 통신 프로그램 ---
프로그램이 실행되면 핸드폰으로 '파이썬에서 값을 입력 하세요' 메시지를 보냅니다.
이후 터미널에 직접 메시지를 입력하면 핸드폰으로 전송됩니다.
핸드폰 앱에서 스위치를 ON/OFF하면 터미널에 상태가 표시됩니다.

MQTT 브로커 '192.168.3.45:1884'에 연결 시도 중...
✔ MQTT 브로커에 성공적으로 연결되었습니다.
✔ 'phone/status' 토픽을 구독합니다. 핸드폰 스위치 상태를 여기서 받습니다.


C:\Users\user\AppData\Local\Temp\ipykernel_18780\1468570504.py:53: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client(client_id=mqtt_client_id)



✔ 초기 메시지 전송: '파이썬에서 값을 입력 하세요'

✔ 핸드폰 클라이언트에게 메시지 전송:
  - 토픽: 'phone/message'
  - 내용: 'ㅏㅓ'
------------------------------

✔ 핸드폰 클라이언트에게 메시지 전송:
  - 토픽: 'phone/message'
  - 내용: 'JA'
------------------------------

⭕ 스위치가 off 되었습니다
------------------------------

✔ 핸드폰 클라이언트에게 메시지 전송:
  - 토픽: 'phone/message'
  - 내용: 'JTR'
------------------------------

✔ 핸드폰 클라이언트에게 메시지 전송:
  - 토픽: 'phone/message'
  - 내용: 'DHKJDS'
------------------------------

💡 스위치가 on 되었습니다
------------------------------

⭕ 스위치가 off 되었습니다
------------------------------

✔ 핸드폰 클라이언트에게 메시지 전송:
  - 토픽: 'phone/message'
  - 내용: 'DFKJ'
------------------------------
프로그램을 종료합니다.

연결 종료.


In [7]:
import paho.mqtt.client as mqtt
import time

# --- MQTT 브로커 설정 ---
mqtt_server = "192.168.3.45"
mqtt_port = 1884
mqtt_client_id = "python_communicator_with_user_input"

# 핸드폰으로 메시지를 보내는 토픽 (사용자 입력 메시지 전송용)
topic_to_phone = "phone/message"

# 핸드폰에서 메시지를 받는 토픽 (스위치 상태 수신용)
topic_from_phone = "phone/status"

# --- 콜백 함수 정의 ---

def on_connect(client, userdata, flags, rc):
    """브로커 연결 성공 시 실행되는 함수입니다."""
    if rc == 0:
        print("✔ MQTT 브로커에 성공적으로 연결되었습니다.")
        # 핸드폰 클라이언트가 보낸 메시지를 받기 위해 토픽을 구독합니다.
        client.subscribe(topic_from_phone)
        print(f"✔ '{topic_from_phone}' 토픽을 구독합니다. 핸드폰 스위치 상태를 여기서 받습니다.")
    else:
        print(f"❌ 연결 실패! 반환 코드(rc): {rc}")

def on_message(client, userdata, msg):
    """메시지를 수신하면 실행되는 함수입니다."""
    if msg.topic == topic_from_phone:
        # 메시지 내용을 문자열로 디코딩
        message = msg.payload.decode("utf-8")

        # 메시지 내용에 따라 정확히 요청하신 문장 출력
        if message.lower() == "on":
            print(f"\n💡 스위치가 on 되었습니다")
        elif message.lower() == "off":
            print(f"\n⭕ 스위치가 off 되었습니다")
        else:
            print(f"\n✔ 핸드폰 클라이언트로부터 메시지 수신:")
            print(f"  - 토픽: {msg.topic}")
            print(f"  - 내용: {message}")
        print("-" * 30)


# --- 메인 프로그램 실행 ---
print("--- 사용자 입력 & MQTT 통신 프로그램 ---")
print("프로그램이 실행되면 핸드폰으로 '파이썬에서 값을 입력 하세요' 메시지를 보냅니다.")
print("이후 터미널에 직접 메시지를 입력하면 핸드폰으로 전송됩니다.")
print("핸드폰 앱에서 스위치를 ON/OFF하면 터미널에 상태가 표시됩니다.")

try:
    # MQTT 클라이언트 생성
    client = mqtt.Client(client_id=mqtt_client_id)
    client.on_connect = on_connect
    client.on_message = on_message

    # 브로커에 연결
    print(f"\nMQTT 브로커 '{mqtt_server}:{mqtt_port}'에 연결 시도 중...")
    client.connect(mqtt_server, mqtt_port, 600)

    # 논블로킹 방식으로 메시지 수신 대기 시작
    client.loop_start()

    # 연결이 완료될 때까지 잠시 대기
    while not client.is_connected():
        time.sleep(1)

    # 초기 메시지 전송
    initial_message = "파이썬에서 값을 입력 하세요"
    client.publish(topic_to_phone, initial_message)
    print(f"\n✔ 초기 메시지 전송: '{initial_message}'")

    # 무한 루프를 돌며 사용자로부터 입력 받기
    while True:
        user_input = input("핸드폰으로 보낼 메시지를 입력하세요 (종료하려면 'quit' 입력): ")

        if user_input.lower() == 'quit':
            print("프로그램을 종료합니다.")
            break

        print(f"\n✔ 핸드폰 클라이언트에게 메시지 전송:")
        print(f"  - 토픽: '{topic_to_phone}'")
        print(f"  - 내용: '{user_input}'")

        # 사용자 입력 메시지 발행
        client.publish(topic_to_phone, user_input)
        print("-" * 30)

except Exception as e:
    print(f"❌ 예외 발생: {e}")

finally:
    # 연결 종료
    if client.is_connected():
        client.loop_stop()
        client.disconnect()
    print("\n연결 종료.")

--- 사용자 입력 & MQTT 통신 프로그램 ---
프로그램이 실행되면 핸드폰으로 '파이썬에서 값을 입력 하세요' 메시지를 보냅니다.
이후 터미널에 직접 메시지를 입력하면 핸드폰으로 전송됩니다.
핸드폰 앱에서 스위치를 ON/OFF하면 터미널에 상태가 표시됩니다.

MQTT 브로커 '192.168.3.45:1884'에 연결 시도 중...
✔ MQTT 브로커에 성공적으로 연결되었습니다.
✔ 'phone/status' 토픽을 구독합니다. 핸드폰 스위치 상태를 여기서 받습니다.


C:\Users\user\AppData\Local\Temp\ipykernel_18780\1074519712.py:53: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client(client_id=mqtt_client_id)



✔ 초기 메시지 전송: '파이썬에서 값을 입력 하세요'

✔ 핸드폰 클라이언트에게 메시지 전송:
  - 토픽: 'phone/message'
  - 내용: 'KKK'
------------------------------

✔ 핸드폰 클라이언트에게 메시지 전송:
  - 토픽: 'phone/message'
  - 내용: 'KJLL'
------------------------------
프로그램을 종료합니다.

연결 종료.


In [ ]:
import paho.mqtt.client as mqtt
import time

# =========================================================
# MQTT 브로커 설정
# =========================================================
MQTT_SERVER = "192.168.3.45"
MQTT_PORT = 1884

MQTT_CLIENT_ID = "python_communicator"

# Python → 핸드폰
TOPIC_TO_PHONE = "phone/message"

# 핸드폰 → Python
TOPIC_FROM_PHONE = "phone/status"


# =========================================================
# MQTT 연결 완료 콜백 함수
# =========================================================
def on_connect(client, userdata, flags, rc):

    if rc == 0:
        print("\n✔ MQTT 브로커 연결 성공", flush=True)
        print(
            f"Broker = {MQTT_SERVER}:{MQTT_PORT}",
            flush=True
        )

        # 핸드폰에서 보내는 스위치 상태를 받기 위해 구독
        client.subscribe(TOPIC_FROM_PHONE)

        print(
            f"✔ Subscribe Topic = {TOPIC_FROM_PHONE}",
            flush=True
        )

    else:
        print(
            f"\n❌ MQTT 연결 실패 : rc={rc}",
            flush=True
        )


# =========================================================
# MQTT 메시지 수신 콜백 함수
# =========================================================
def on_message(client, userdata, msg):

    try:
        # bytes → 문자열
        message = msg.payload.decode("utf-8")

        print("\n" + "=" * 40, flush=True)
        print("📱 핸드폰 MQTT 메시지 수신", flush=True)
        print(f"Topic   : {msg.topic}", flush=True)
        print(f"Payload : {message}", flush=True)

        # -------------------------------------------------
        # IoT MQTT Panel 스위치 ON
        # -------------------------------------------------
        if message.strip().lower() == "on":

            print(
                "💡 스위치가 ON 되었습니다.",
                flush=True
            )

        # -------------------------------------------------
        # IoT MQTT Panel 스위치 OFF
        # -------------------------------------------------
        elif message.strip().lower() == "off":

            print(
                "⭕ 스위치가 OFF 되었습니다.",
                flush=True
            )

        # -------------------------------------------------
        # 기타 메시지
        # -------------------------------------------------
        else:

            print(
                f"✔ 기타 메시지 수신 : {message}",
                flush=True
            )

        print("=" * 40, flush=True)

    except Exception as e:

        print(
            f"❌ 메시지 처리 오류 : {e}",
            flush=True
        )


# =========================================================
# 메인 프로그램
# =========================================================
client = None

try:

    print("=" * 50)
    print("Python ↔ IoT MQTT Panel 통신 프로그램")
    print("=" * 50)

    # -----------------------------------------------------
    # MQTT 클라이언트 생성
    # -----------------------------------------------------
    client = mqtt.Client(
        client_id=MQTT_CLIENT_ID
    )

    # 콜백 함수 등록
    client.on_connect = on_connect
    client.on_message = on_message

    # -----------------------------------------------------
    # MQTT Broker 연결
    # -----------------------------------------------------
    print(
        f"\nMQTT Broker 연결 중..."
        f" {MQTT_SERVER}:{MQTT_PORT}"
    )

    client.connect(
        MQTT_SERVER,
        MQTT_PORT,
        60
    )

    # -----------------------------------------------------
    # MQTT 네트워크 루프 시작
    # -----------------------------------------------------
    # 별도의 스레드에서 MQTT 메시지를 계속 수신
    client.loop_start()

    # -----------------------------------------------------
    # 연결 완료 대기
    # -----------------------------------------------------
    print("MQTT 연결 완료 대기 중...")

    while not client.is_connected():
        time.sleep(0.1)

    # -----------------------------------------------------
    # 초기 메시지 전송
    # -----------------------------------------------------
    initial_message = "파이썬에서 값을 입력 하세요"

    client.publish(
        TOPIC_TO_PHONE,
        initial_message
    )

    print("\n✔ 초기 메시지 전송")
    print(f"Topic   : {TOPIC_TO_PHONE}")
    print(f"Payload : {initial_message}")

    print("\n" + "=" * 50)
    print("핸드폰 스위치를 ON/OFF하면")
    print("Python에서 즉시 상태를 확인할 수 있습니다.")
    print("=" * 50)

    # -----------------------------------------------------
    # 사용자 입력 → 핸드폰 전송
    # -----------------------------------------------------
    while True:

        print()

        user_input = input(
            "핸드폰으로 보낼 메시지 "
            "(종료: quit) : "
        )

        # -------------------------------------------------
        # 프로그램 종료
        # -------------------------------------------------
        if user_input.strip().lower() == "quit":

            print("\n프로그램을 종료합니다.")
            break

        # -------------------------------------------------
        # 빈 문자열 검사
        # -------------------------------------------------
        if user_input.strip() == "":

            print(
                "⚠ 메시지를 입력해 주세요.",
                flush=True
            )

            continue

        # -------------------------------------------------
        # 핸드폰으로 메시지 전송
        # -------------------------------------------------
        result = client.publish(
            TOPIC_TO_PHONE,
            user_input
        )

        print("\n✔ 핸드폰으로 메시지 전송")
        print(f"Topic   : {TOPIC_TO_PHONE}")
        print(f"Payload : {user_input}")
        print("-" * 40)


# =========================================================
# 예외 처리
# =========================================================
except KeyboardInterrupt:

    print("\nCtrl+C 입력")
    print("프로그램을 종료합니다.")


except Exception as e:

    print(
        f"\n❌ 프로그램 오류 : {e}"
    )


# =========================================================
# 프로그램 종료 처리
# =========================================================
finally:

    if client is not None:

        try:

            client.loop_stop()
            client.disconnect()

        except:
            pass

    print("\nMQTT 연결 종료")

In [ ]:
# 핸드폰에서 숫자를 받는 토픽
# 핸드폰 클라이언트 앱에서 이 토픽으로 0~255 사이의 값을 보내야 합니다.
import paho.mqtt.client as mqtt
import time

# --- MQTT 브로커 설정 ---
mqtt_server = "192.168.3.45"
mqtt_port = 1884
mqtt_client_id = "python_communicator_with_user_input"

topic_from_phone = "phone/number_value"

# --- 콜백 함수 정의 ---

def on_connect(client, userdata, flags, rc):
    """브로커 연결 성공 시 실행되는 함수입니다."""
    if rc == 0:
        print("✔ MQTT 브로커에 성공적으로 연결되었습니다.")
        # 핸드폰 클라이언트가 보낼 값을 받기 위해 토픽을 구독합니다.
        client.subscribe(topic_from_phone)
        print(f"✔ '{topic_from_phone}' 토픽을 구독합니다. 핸드폰에서 오는 값을 기다립니다.")
    else:
        print(f"❌ 연결 실패! 반환 코드(rc): {rc}")

def on_message(client, userdata, msg):
    """메시지를 수신하면 실행되는 함수입니다."""
    if msg.topic == topic_from_phone:
        try:
            # 메시지 내용을 정수로 변환
            received_value = int(msg.payload.decode("utf-8"))

            # 값의 범위를 확인하고 출력
            if received_value > 20:
                print(f"\n✅ 수신된 값 {received_value}은(는) 20보다 큽니다.")
            else:
                print(f"\n❌ 수신된 값 {received_value}은(는) 20보다 작습니다.")

        except ValueError:
            # 숫자로 변환할 수 없는 메시지가 오면 오류 출력
            print(f"\n⚠️ 경고: 숫자가 아닌 값이 수신되었습니다: {msg.payload.decode('utf-8')}")
        print("-" * 30)


# --- 메인 프로그램 실행 ---
print("--- 핸드폰 값 수신 및 판별 프로그램 ---")
print("이 프로그램은 핸드폰에서 보내는 0~255 사이의 값을 받습니다.")
print(f"'{topic_from_phone}' 토픽으로 값을 보내면 터미널에 결과가 표시됩니다.")

try:
    # MQTT 클라이언트 생성
    client = mqtt.Client(client_id=mqtt_client_id)
    client.on_connect = on_connect
    client.on_message = on_message

    # 브로커에 연결
    print(f"\nMQTT 브로커 '{mqtt_server}:{mqtt_port}'에 연결 시도 중...")
    client.connect(mqtt_server, mqtt_port, 600)

    # 논블로킹 방식으로 메시지 수신 대기 시작
    client.loop_start()

    # 메시지를 받을 수 있도록 무한 대기
    print("\n핸드폰 클라이언트로부터 값을 기다리는 중... (프로그램 종료하려면 Ctrl+C)")
    while True:
        time.sleep(1)

except KeyboardInterrupt:
    print("\n프로그램을 수동으로 종료합니다.")
except Exception as e:
    print(f"❌ 예외 발생: {e}")

finally:
    # 연결 종료
    if client.is_connected():
        client.loop_stop()
        client.disconnect()
    print("\n연결 종료.")

--- 핸드폰 값 수신 및 판별 프로그램 ---
이 프로그램은 핸드폰에서 보내는 0~255 사이의 값을 받습니다.
'phone/number_value' 토픽으로 값을 보내면 터미널에 결과가 표시됩니다.

MQTT 브로커 '192.168.3.45:1884'에 연결 시도 중...

핸드폰 클라이언트로부터 값을 기다리는 중... (프로그램 종료하려면 Ctrl+C)
✔ MQTT 브로커에 성공적으로 연결되었습니다.
✔ 'phone/number_value' 토픽을 구독합니다. 핸드폰에서 오는 값을 기다립니다.


C:\Users\user\AppData\Local\Temp\ipykernel_18780\2532614617.py:51: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client(client_id=mqtt_client_id)



❌ 수신된 값 20은(는) 20보다 작습니다.
------------------------------

✅ 수신된 값 50은(는) 20보다 큽니다.
------------------------------

✅ 수신된 값 570은(는) 20보다 큽니다.
------------------------------


In [ ]:
# combobox 활용 프로그램
import paho.mqtt.client as mqtt
import time

# --- MQTT 브로커 설정 ---
mqtt_server = "192.168.219.101"
mqtt_port = 1883
mqtt_client_id = "python_combobox_subscriber"

# IoT MQTT Pannel 앱의 콤보 박스에서 사용하는 토픽
topic_from_combobox = "select_options"

# --- 콜백 함수 정의 ---

def on_connect(client, userdata, flags, rc):
    """브로커 연결 성공 시 실행되는 함수입니다."""
    if rc == 0:
        print("✔ MQTT 브로커에 성공적으로 연결되었습니다.")
        # 콤보 박스 위젯의 토픽을 구독합니다.
        client.subscribe(topic_from_combobox)
        print(f"✔ '{topic_from_combobox}' 토픽을 구독합니다. 콤보 박스에서 오는 메시지를 기다립니다.")
    else:
        print(f"❌ 연결 실패! 반환 코드(rc): {rc}")

def on_message(client, userdata, msg):
    """콤보 박스에서 메시지를 수신하면 실행되는 함수입니다."""
    if msg.topic == topic_from_combobox:
        # 메시지 내용을 문자열로 디코딩
        selected_option = msg.payload.decode("utf-8")

        print(f"\n✅ 콤보 박스에서 선택된 내용 수신:")
        print(f"  - 수신된 토픽: {msg.topic}")
        print(f"  - 선택된 내용: {selected_option}")

        # 선택된 내용에 따라 추가적인 동작을 수행할 수 있습니다.
        # if selected_option == "LED ON":
        #     print(">> LED를 켭니다.")
        # elif selected_option == "FAN OFF":
        #     print(">> FAN을 끕니다.")

        print("-" * 30)

# --- 메인 프로그램 실행 ---
print("--- 콤보 박스 내용 수신 프로그램 ---")
print("IoT MQTT Pannel 앱에서 'select_options' 토픽으로 메시지를 보내세요.")

try:
    # MQTT 클라이언트 생성
    client = mqtt.Client(client_id=mqtt_client_id)
    client.on_connect = on_connect
    client.on_message = on_message

    # 브로커에 연결
    print(f"\nMQTT 브로커 '{mqtt_server}:{mqtt_port}'에 연결 시도 중...")
    client.connect(mqtt_server, mqtt_port, 600)

    # 논블로킹 방식으로 메시지 수신 대기 시작
    client.loop_start()

    # 메시지를 받을 수 있도록 무한 대기
    print("\n콤보 박스에서 선택된 내용을 기다리는 중... (프로그램 종료하려면 Ctrl+C)")
    while True:
        time.sleep(1)

except KeyboardInterrupt:
    print("\n프로그램을 수동으로 종료합니다.")
except Exception as e:
    print(f"❌ 예외 발생: {e}")

finally:
    # 연결 종료
    if client.is_connected():
        client.loop_stop()
        client.disconnect()
    print("\n연결 종료.")


In [ ]:
import paho.mqtt.client as mqtt
import time

# --- MQTT 브로커 설정 ---
mqtt_server = "192.168.219.101"
mqtt_port = 1883
mqtt_client_id = "python_radiobutton_subscriber"

# IoT MQTT Pannel 앱의 라디오 버튼에서 사용하는 토픽
topic_from_radiobuttons = "radio_options"

# --- 콜백 함수 정의 ---

def on_connect(client, userdata, flags, rc):
    """브로커 연결 성공 시 실행되는 함수입니다."""
    if rc == 0:
        print("✔ MQTT 브로커에 성공적으로 연결되었습니다.")
        # 라디오 버튼 위젯의 토픽을 구독합니다.
        client.subscribe(topic_from_radiobuttons)
        print(f"✔ '{topic_from_radiobuttons}' 토픽을 구독합니다. 라디오 버튼에서 오는 메시지를 기다립니다.")
    else:
        print(f"❌ 연결 실패! 반환 코드(rc): {rc}")

def on_message(client, userdata, msg):
    """라디오 버튼에서 메시지를 수신하면 실행되는 함수입니다."""
    if msg.topic == topic_from_radiobuttons:
        # 메시지 내용을 문자열로 디코딩
        selected_option = msg.payload.decode("utf-8")

        print(f"\n✅ 라디오 버튼에서 선택된 내용 수신:")
        print(f"  - 수신된 토픽: {msg.topic}")
        print(f"  - 선택된 내용: {selected_option}")

        # 선택된 내용에 따라 추가적인 동작을 수행할 수 있습니다.
        if selected_option == "on":
             print(">> LED를 켭니다.")
        elif selected_option == "off":
             print(">> FAN을 끕니다.")

        print("-" * 30)

# --- 메인 프로그램 실행 ---
print("--- 라디오 버튼 내용 수신 프로그램 ---")
print("IoT MQTT Pannel 앱에서 'radio_options' 토픽으로 메시지를 보내세요.")

try:
    # MQTT 클라이언트 생성
    client = mqtt.Client(client_id=mqtt_client_id)
    client.on_connect = on_connect
    client.on_message = on_message

    # 브로커에 연결
    print(f"\nMQTT 브로커 '{mqtt_server}:{mqtt_port}'에 연결 시도 중...")
    client.connect(mqtt_server, mqtt_port, 600)

    # 논블로킹 방식으로 메시지 수신 대기 시작
    client.loop_start()

    # 메시지를 받을 수 있도록 무한 대기
    print("\n라디오 버튼에서 선택된 내용을 기다리는 중... (프로그램 종료하려면 Ctrl+C)")
    while True:
        time.sleep(1)

except KeyboardInterrupt:
    print("\n프로그램을 수동으로 종료합니다.")
except Exception as e:
    print(f"❌ 예외 발생: {e}")

finally:
    # 연결 종료
    if client.is_connected():
        client.loop_stop()
        client.disconnect()
    print("\n연결 종료.")


In [ ]:
import paho.mqtt.client as mqtt
import time
mqtt_server = "192.168.0.12"
mqtt_port = 1884
mqtt_client_id = "python_led_publisher"
topic_led1 = "led/status/1"
topic_led2 = "led/status/2"
def on_connect(client, userdata, flags, rc):
    """브로커 연결 성공 시 실행되는 함수입니다."""
    if rc == 0:
        print("✔ MQTT 브로커에 성공적으로 연결되었습니다.")
    else:
        print(f"❌ 연결 실패! 반환 코드(rc): {rc}")

# on_message 함수는 메시지를 받는 역할이므로, 이 프로그램에서는 필요하지 않습니다.
# def on_message(client, userdata, msg):
#     pass

# --- 메인 프로그램 실행 ---
print("--- IoT MQTT Pannel LED 제어 프로그램 ---")
print("1: LED 1 ON (IoT Pannel에 '1' 보냄)")
print("0: LED 1 OFF (IoT Pannel에 '0' 보냄)")
print("2: LED 2 ON (IoT Pannel에 '1' 보냄)")
print("3: LED 2 OFF (IoT Pannel에 '0' 보냄)")
print("---------------------------------------------")

try:
    # MQTT 클라이언트 생성
    client = mqtt.Client(client_id=mqtt_client_id)
    client.on_connect = on_connect
    # 브로커에 연결
    print(f"\nMQTT 브로커 '{mqtt_server}:{mqtt_port}'에 연결 시도 중...")
    client.connect(mqtt_server, mqtt_port, 600)
    # 논블로킹 방식으로 메시지 발행 가능하게 설정
    client.loop_start()
    # 연결이 완료될 때까지 잠시 대기
    while not client.is_connected():
        time.sleep(1)
    while True:
        user_input = input("제어 번호를 입력하세요 (종료하려면 'quit' 입력): ")

        if user_input.lower() == 'quit':
            print("프로그램을 종료합니다.")
            break

        message_to_send = ""
        topic_to_publish = ""

        if user_input == "1":
            topic_to_publish = topic_led1
            message_to_send = "1"
            print("LED 1을 ON 시킵니다.")
        elif user_input == "0":
            topic_to_publish = topic_led1
            message_to_send = "0"
            print("LED 1을 OFF 시킵니다.")
        elif user_input == "2":
            topic_to_publish = topic_led2
            message_to_send = "1"
            print("LED 2를 ON 시킵니다.")
        elif user_input == "3":
            topic_to_publish = topic_led2
            message_to_send = "0"
            print("LED 2를 OFF 시킵니다.")
        else:
            print("❌ 잘못된 입력입니다. 1, 0, 2, 3 중 하나를 입력해주세요.")
            continue

        # MQTT 메시지 발행
        if topic_to_publish and message_to_send:
            client.publish(topic_to_publish, message_to_send)
            print(f"✔ '{topic_to_publish}' 토픽으로 메시지 '{message_to_send}' 발행")
        print("-" * 30)

except Exception as e:
    print(f"❌ 예외 발생: {e}")

finally:
    # 연결 종료
    if client.is_connected():
        client.loop_stop()
        client.disconnect()
    print("\n연결 종료.")
    
# import paho.mqtt.client as mqtt
# client = mqtt.Client("python_led_publisher")
# → 즉, 파이썬 프로그램이 MQTT 네트워크에 참여하기 위한 클라이언트입니다.
# 이 클라이언트는
# 브로커(MQTT 서버)에 연결하고 (client.connect(...))
# 메시지를 발행하거나(client.publish(...))
# 구독할 수도 있습니다 (client.subscribe(...)).
# client.publish()
# 이 메서드는 메시지를 보내는(발행하는) 기능입니다.
# client.publish(topic, payload, qos=0, retain=False)
# 인자	의미
# topic	메시지를 보낼 “주제 이름” (예: "led/status/1")
# payload	보낼 실제 메시지 내용 (예: "ON" 또는 "OFF")
# qos	품질 수준 (기본 0, 선택적)
# retain	메시지를 브로커에 저장할지 여부 (기본 False)    
# 이 파이썬 프로그램이 publisher
# MQTT 브로커가 중간 전달자
# 구독 중인 다른 장치(예: 아두이노, ESP32, 다른 PC)가 subscriber이 됩니다.
# 요약하자면
# client.publish(...) 의 “client”는 현재 코드 안에서 MQTT 브로커에 연결한
# 파이썬 클라이언트 자신입니다.
    
    
    
    
    

In [ ]:
#multi-state indicator

import paho.mqtt.client as mqtt
import time

# --- MQTT 브로커 설정 ---
mqtt_server = "192.168.0.12"
mqtt_port = 1884
mqtt_client_id = "python_state_publisher"

# Multi-State Indicator 위젯 제어용 토픽
topic_to_indicator = "device/status"

# --- 콜백 함수 정의 ---

def on_connect(client, userdata, flags, rc):
    """브로커 연결 성공 시 실행되는 함수입니다."""
    if rc == 0:
        print("✔ MQTT 브로커에 성공적으로 연결되었습니다.")
    else:
        print(f"❌ 연결 실패! 반환 코드(rc): {rc}")

# --- 메인 프로그램 실행 ---
print("--- Multi-State Indicator 제어 프로그램 ---")
print("0: 대기 상태 (Gray)")
print("1: 작동 중 상태 (Green)")
print("2: 오류 상태 (Red)")
print("---------------------------------------------")

try:
    # MQTT 클라이언트 생성
    client = mqtt.Client(client_id=mqtt_client_id)
    client.on_connect = on_connect

    # 브로커에 연결
    print(f"\nMQTT 브로커 '{mqtt_server}:{mqtt_port}'에 연결 시도 중...")
    client.connect(mqtt_server, mqtt_port, 600)

    # 논블로킹 방식으로 메시지 발행 가능하게 설정
    client.loop_start()

    # 연결이 완료될 때까지 잠시 대기
    while not client.is_connected():
        time.sleep(1)

    # 무한 루프를 돌며 사용자로부터 입력 받기
    while True:
        user_input = input("상태 번호를 입력하세요 (종료하려면 'quit' 입력): ")

        if user_input.lower() == 'quit':
            print("프로그램을 종료합니다.")
            break

        message_to_send = ""

        if user_input == "0":
            message_to_send = "0"
            print("대기 상태를 보냅니다.")
        elif user_input == "1":
            message_to_send = "1"
            print("작동 중 상태를 보냅니다.")
        elif user_input == "2":
            message_to_send = "2"
            print("오류 상태를 보냅니다.")
        else:
            print("❌ 잘못된 입력입니다. 0, 1, 2 중 하나를 입력해주세요.")
            continue

        # MQTT 메시지 발행
        client.publish(topic_to_indicator, message_to_send)
        print(f"✔ '{topic_to_indicator}' 토픽으로 메시지 '{message_to_send}' 발행")
        print("-" * 30)

except Exception as e:
    print(f"❌ 예외 발생: {e}")

finally:
    # 연결 종료
    if client.is_connected():
        client.loop_stop()
        client.disconnect()
    print("\n연결 종료.")

In [ ]:
import paho.mqtt.client as mqtt
import time

# --- MQTT 브로커 설정 ---
mqtt_server = "192.168.0.12"
mqtt_port = 1884
mqtt_client_id = "python_progress_publisher"

# Progress Bar 위젯 제어용 토픽
topic_to_progress = "progress/status"

# --- 콜백 함수 정의 ---

def on_connect(client, userdata, flags, rc):
    """브로커 연결 성공 시 실행되는 함수입니다."""
    if rc == 0:
        print("✔ MQTT 브로커에 성공적으로 연결되었습니다.")
    else:
        print(f"❌ 연결 실패! 반환 코드(rc): {rc}")

# --- 메인 프로그램 실행 ---
print("--- Progress Bar 제어 프로그램 ---")
print("0부터 100까지의 진행률을 1초마다 IoT MQTT Pannel로 보냅니다.")
print("---------------------------------------")

try:
    # MQTT 클라이언트 생성
    client = mqtt.Client(client_id=mqtt_client_id)
    client.on_connect = on_connect

    # 브로커에 연결
    print(f"\nMQTT 브로커 '{mqtt_server}:{mqtt_port}'에 연결 시도 중...")
    client.connect(mqtt_server, mqtt_port, 600)

    # 논블로킹 방식으로 메시지 발행 가능하게 설정
    client.loop_start()

    # 연결이 완료될 때까지 잠시 대기
    while not client.is_connected():
        time.sleep(1)

    # 0부터 100까지 1초 간격으로 진행률 발행
    for i in range(101):
        progress_value = str(i)

        # MQTT 메시지 발행
        client.publish(topic_to_progress, progress_value)
        print(f"✔ '{topic_to_progress}' 토픽으로 진행률 '{progress_value}' 발행")

        # 1초 대기
        time.sleep(0.5)

    print("\n✅ 모든 진행률 전송 완료!")

except Exception as e:
    print(f"❌ 예외 발생: {e}")

finally:
    # 연결 종료
    if client.is_connected():
        client.loop_stop()
        client.disconnect()
    print("\n연결 종료.")

In [ ]:

#gauge사용
import paho.mqtt.client as mqtt
import time
import random

# --- MQTT 브로커 설정 ---
mqtt_server = "192.168.223.97"
mqtt_port = 1883
mqtt_client_id = "python_gauge_publisher"

# Gauge 위젯 제어용 토픽
topic_to_gauge = "sensor/temperature"

# --- 콜백 함수 정의 ---

def on_connect(client, userdata, flags, rc):
    """브로커 연결 성공 시 실행되는 함수입니다."""
    if rc == 0:
        print("✔ MQTT 브로커에 성공적으로 연결되었습니다.")
    else:
        print(f"❌ 연결 실패! 반환 코드(rc): {rc}")

# --- 메인 프로그램 실행 ---
print("--- Gauge 위젯 제어 프로그램 ---")
print("0~50 사이의 임의의 값을 1초마다 IoT MQTT Pannel로 보냅니다.")
print("---------------------------------------")

try:
    # MQTT 클라이언트 생성
    client = mqtt.Client(client_id=mqtt_client_id)
    client.on_connect = on_connect

    # 브로커에 연결
    print(f"\nMQTT 브로커 '{mqtt_server}:{mqtt_port}'에 연결 시도 중...")
    client.connect(mqtt_server, mqtt_port, 600)

    # 논블로킹 방식으로 메시지 발행 가능하게 설정
    client.loop_start()

    # 연결이 완료될 때까지 잠시 대기
    while not client.is_connected():
        time.sleep(1)

    # 무한 루프를 돌며 임의의 값 발행
    while True:
        # 0.0부터 50.0까지의 임의의 실수 값 생성
        temperature_value = round(random.uniform(0, 50), 1)

        # MQTT 메시지 발행
        client.publish(topic_to_gauge, str(temperature_value))
        print(f"✔ '{topic_to_gauge}' 토픽으로 값 '{temperature_value}' 발행")

        # 1초 대기
        time.sleep(1)

except KeyboardInterrupt:
    print("\n프로그램을 수동으로 종료합니다.")
except Exception as e:
    print(f"❌ 예외 발생: {e}")

finally:
    # 연결 종료
    if client.is_connected():
        client.loop_stop()
        client.disconnect()
    print("\n연결 종료.")

In [ ]:
#color picker 파이썬에서 핸드폰으로 보내기
import paho.mqtt.client as mqtt
import time

# --- MQTT 브로커 설정 ---
mqtt_server = "192.168.223.110"
mqtt_port = 1883
mqtt_client_id = "python_color_publisher"

# Color Picker 위젯 제어용 토픽
topic_to_color_picker = "device/color"

# --- 콜백 함수 정의 ---

def on_connect(client, userdata, flags, rc):
    """브로커 연결 성공 시 실행되는 함수입니다."""
    if rc == 0:
        print("✔ MQTT 브로커에 성공적으로 연결되었습니다.")
    else:
        print(f"❌ 연결 실패! 반환 코드(rc): {rc}")

# --- 메인 프로그램 실행 ---
print("--- Color Picker 제어 프로그램 ---")
print("R(0-255), G(0-255), B(0-255) 값을 입력하여 색상을 전송합니다.")
print("---------------------------------------")

try:
    # MQTT 클라이언트 생성
    client = mqtt.Client(client_id=mqtt_client_id)
    client.on_connect = on_connect

    # 브로커에 연결
    print(f"\nMQTT 브로커 '{mqtt_server}:{mqtt_port}'에 연결 시도 중...")
    client.connect(mqtt_server, mqtt_port, 600)

    # 논블로킹 방식으로 메시지 발행 가능하게 설정
    client.loop_start()

    # 연결이 완료될 때까지 잠시 대기
    while not client.is_connected():
        time.sleep(1)

    # 무한 루프를 돌며 사용자로부터 입력 받기
    while True:
        try:
            r_value = int(input("Red 값 (0-255)을 입력하세요 (종료: -1): "))
            if r_value == -1:
                print("프로그램을 종료합니다.")
                break

            g_value = int(input("Green 값 (0-255)을 입력하세요: "))
            b_value = int(input("Blue 값 (0-255)을 입력하세요: "))

            # 값의 유효성 검사
            if not (0 <= r_value <= 255 and 0 <= g_value <= 255 and 0 <= b_value <= 255):
                print("❌ 값의 범위가 잘못되었습니다. 0-255 사이의 값을 입력해주세요.")
                continue

            # RGB 값을 'R,G,B' 형식의 문자열로 조합
            color_message = f"{r_value},{g_value},{b_value}"

            # MQTT 메시지 발행
            client.publish(topic_to_color_picker, color_message)
            print(f"✔ '{topic_to_color_picker}' 토픽으로 색상 '{color_message}' 발행")

        except ValueError:
            print("❌ 잘못된 입력입니다. 숫자를 입력해주세요.")

        print("-" * 30)

except Exception as e:
    print(f"❌ 예외 발생: {e}")

finally:
    # 연결 종료
    if client.is_connected():
        client.loop_stop()
        client.disconnect()
    print("\n연결 종료.")


In [ ]:
#핸드폰 에서 색상을 파이썬으로 보내는 방법(ok)
import paho.mqtt.client as mqtt
import time

# --- MQTT 브로커 설정 ---
mqtt_server = "192.168.223.97"
mqtt_port = 1883
mqtt_client_id = "python_color_subscriber"

# Color Picker 위젯의 토픽 (수신용)
topic_from_color_picker = "device/color"

# --- 콜백 함수 정의 ---

def on_connect(client, userdata, flags, rc):
    """브로커 연결 성공 시 실행되는 함수입니다."""
    if rc == 0:
        print("✔ MQTT 브로커에 성공적으로 연결되었습니다.")
        # Color Picker 위젯의 토픽을 구독합니다.
        client.subscribe(topic_from_color_picker)
        print(f"✔ '{topic_from_color_picker}' 토픽을 구독합니다. 색상 값을 기다립니다.")
    else:
        print(f"❌ 연결 실패! 반환 코드(rc): {rc}")

def on_message(client, userdata, msg):
    """Color Picker 위젯에서 메시지를 수신하면 실행되는 함수입니다."""
    if msg.topic == topic_from_color_picker:
        try:
            # 메시지 내용을 문자열로 디코딩
            rgb_string = msg.payload.decode("utf-8")

            # 쉼표를 기준으로 RGB 값을 분리하여 정수로 변환
            r, g, b = map(int, rgb_string.split(','))

            print(f"\n✅ 색상 값 수신:")
            print(f"  - 토픽: {msg.topic}")
            print(f"  - RGB 값: ({r}, {g}, {b})")

            # 추가적인 색상 처리 로직을 여기에 구현할 수 있습니다.
            # 예: 특정 색상에 따른 동작 수행
            if r > 200 and g < 50 and b < 50:
                print("  >> 빨간색 계열이 선택되었습니다.")
            elif r < 50 and g > 200 and b < 50:
                print("  >> 초록색 계열이 선택되었습니다.")

        except (ValueError, IndexError):
            print(f"\n❌ 잘못된 형식의 값이 수신되었습니다: {msg.payload.decode('utf-8')}")

        print("-" * 30)

# --- 메인 프로그램 실행 ---
print("--- Color Picker 값 수신 프로그램 ---")
print("IoT MQTT Pannel 앱에서 Color Picker 위젯을 통해 색상을 선택하세요.")

try:
    # MQTT 클라이언트 생성
    client = mqtt.Client(client_id=mqtt_client_id)
    client.on_connect = on_connect
    client.on_message = on_message

    # 브로커에 연결
    print(f"\nMQTT 브로커 '{mqtt_server}:{mqtt_port}'에 연결 시도 중...")
    client.connect(mqtt_server, mqtt_port, 600)

    # 논블로킹 방식으로 메시지 수신 대기 시작
    client.loop_start()

    # 메시지를 받을 수 있도록 무한 대기
    print("\n색상 값을 기다리는 중... (프로그램 종료하려면 Ctrl+C)")
    while True:
        time.sleep(1)

except KeyboardInterrupt:
    print("\n프로그램을 수동으로 종료합니다.")
except Exception as e:
    print(f"❌ 예외 발생: {e}")

finally:
    # 연결 종료
    if client.is_connected():
        client.loop_stop()
        client.disconnect()
    print("\n연결 종료.")



In [ ]:
#date time picker
import paho.mqtt.client as mqtt
import time
from datetime import datetime

# --- MQTT 브로커 설정 ---
mqtt_server = "192.168.223.97"
mqtt_port = 1883
mqtt_client_id = "python_datetime_publisher"

# Date & Time Picker 위젯 제어용 토픽
topic_to_datetime_picker = "schedule/datetime"

# --- 콜백 함수 정의 ---

def on_connect(client, userdata, flags, rc):
    """브로커 연결 성공 시 실행되는 함수입니다."""
    if rc == 0:
        print("✔ MQTT 브로커에 성공적으로 연결되었습니다.")
    else:
        print(f"❌ 연결 실패! 반환 코드(rc): {rc}")

# --- 메인 프로그램 실행 ---
print("--- Date & Time Picker 제어 프로그램 ---")
print("YYYY-MM-DD HH:MM 형식으로 날짜와 시간을 입력하여 전송합니다.")
print("예시) 2024-05-15 14:30")
print("-----------------------------------------")

try:
    # MQTT 클라이언트 생성
    client = mqtt.Client(client_id=mqtt_client_id)
    client.on_connect = on_connect

    # 브로커에 연결
    print(f"\nMQTT 브로커 '{mqtt_server}:{mqtt_port}'에 연결 시도 중...")
    client.connect(mqtt_server, mqtt_port, 600)

    # 논블로킹 방식으로 메시지 발행 가능하게 설정
    client.loop_start()

    # 연결이 완료될 때까지 잠시 대기
    while not client.is_connected():
        time.sleep(1)

    # 무한 루프를 돌며 사용자로부터 입력 받기
    while True:
        user_input = input("날짜와 시간을 입력하세요 (종료하려면 'quit' 입력): ")

        if user_input.lower() == 'quit':
            print("프로그램을 종료합니다.")
            break

        try:
            # 입력된 문자열을 날짜/시간 객체로 파싱하여 유효성 검사
            datetime_obj = datetime.strptime(user_input, '%Y-%m-%d %H:%M')

            # MQTT 메시지 발행
            client.publish(topic_to_datetime_picker, user_input)
            print(f"✔ '{topic_to_datetime_picker}' 토픽으로 '{user_input}' 발행")

        except ValueError:
            print("❌ 잘못된 입력 형식입니다. 'YYYY-MM-DD HH:MM' 형식으로 입력해주세요.")

        print("-" * 30)

except Exception as e:
    print(f"❌ 예외 발생: {e}")

finally:
    # 연결 종료
    if client.is_connected():
        client.loop_stop()
        client.disconnect()
    print("\n연결 종료.")

In [ ]:
#line graph
 '''데이터를 수신하고, 대시보드 노드를 통해 차트나 그래프로 시각화하는 강력한 도구입니다. 복잡해 보일 수 있지만, 한 번 설정해두면 웹 브라우저를 통Mosquitto를 이용한 확인
Mosquitto 브로커가 설치된 컴퓨터의 명령 프롬프트(CMD)에서 mosquitto_sub 명령어를 사용하여 Wi-Fi 주소로 전송되는 데이터를 확인할 수 있습니다.
Mosquitto 설치 폴더로 이동:
C:\Program Files\mosquitto로 이동합니다.
구독 명령어 입력:
mosquitto_sub -h 192.168.223.110 -t device/status/chart
-h 192.168.223.110: MQTT 브로커가 실행 중인 PC의 Wi-Fi 주소(IP 주소)를 지정합니다.
-t device/status/chart: Chart 위젯의 토픽을 구독합니다.
이 명령어를 실행하면, 스마트폰에서 Chart 위젯을 업데이트할 때마다 터미널에 실시간으로 데이터가 출력되는 것을 확인할 수 있습니다.

대시보드 솔루션 활용
MQTT 데이터를 웹 브라우저나 다른 컴퓨터에서 시각적으로 확인하고 싶다면, MQTT 데이터를 웹 기반 대시보드로 시각화하는 다른 솔루션을 활용할 수 있습니다.
Node-RED: Node-RED는 MQTT 노드를 사용하여해 Wi-Fi 주소로 접속하여 실시간으로 차트를 볼 수 있습니다.

Grafana: MQTT 데이터를 데이터베이스(예: InfluxDB)에 저장하고, Grafana를 연결하여 고급 대시보드를 구축할 수도 있습니다. 이 방법은 데이터를 장기간 보관하고 분석하는 데 유용합니다.

결론적으로, IoT MQTT Pannel 앱 자체만으로는 Wi-Fi 주소로 직접 접속해 차트를 보는 기능은 없지만, mosquitto_sub 명령어나 다른 대시보드 솔루션을 활용하여 같은 Wi-Fi 네트워크에 있는 다른 기기에서 데이터를 확인하고 시각화할 수 있습니다.'''
import paho.mqtt.client as mqtt
import time
import random

# --- MQTT 브로커 설정 ---
mqtt_server = "192.168.223.97"
mqtt_port = 1883
mqtt_client_id = "python_linegraph_publisher"

# Line Graph 위젯 제어용 토픽
topic_temp = "sensor/temperature"
topic_humid = "sensor/humidity"

# --- 콜백 함수 정의 ---

def on_connect(client, userdata, flags, rc):
    """브로커 연결 성공 시 실행되는 함수입니다."""
    if rc == 0:
        print("✔ MQTT 브로커에 성공적으로 연결되었습니다.")
    else:
        print(f"❌ 연결 실패! 반환 코드(rc): {rc}")

# --- 메인 프로그램 실행 ---
print("--- Line Graph 제어 프로그램 ---")
print("온도와 습도 데이터를 1초마다 IoT MQTT Pannel로 보냅니다.")
print("---------------------------------------")

try:
    # MQTT 클라이언트 생성
    client = mqtt.Client(client_id=mqtt_client_id)
    client.on_connect = on_connect

    # 브로커에 연결
    print(f"\nMQTT 브로커 '{mqtt_server}:{mqtt_port}'에 연결 시도 중...")
    client.connect(mqtt_server, mqtt_port, 600)

    # 논블로킹 방식으로 메시지 발행 가능하게 설정
    client.loop_start()

    # 연결이 완료될 때까지 잠시 대기
    while not client.is_connected():
        time.sleep(1)

    # 무한 루프를 돌며 임의의 값 발행
    while True:
        # 0.0부터 50.0까지의 임의의 실수 값 생성 (온도 시뮬레이션)
        temp_value = round(random.uniform(15.0, 30.0), 1)

        # 0.0부터 100.0까지의 임의의 실수 값 생성 (습도 시뮬레이션)
        humid_value = round(random.uniform(40.0, 80.0), 1)

        # MQTT 메시지 발행
        client.publish(topic_temp, str(temp_value))
        client.publish(topic_humid, str(humid_value))

        print(f"✔ 온도 '{temp_value}'와 습도 '{humid_value}'를 발행했습니다.")

        # 1초 대기
        time.sleep(1)

except KeyboardInterrupt:
    print("\n프로그램을 수동으로 종료합니다.")
except Exception as e:
    print(f"❌ 예외 발생: {e}")

finally:
    # 연결 종료
    if client.is_connected():
        client.loop_stop()
        client.disconnect()
    print("\n연결 종료.")

In [ ]:
#chart
import paho.mqtt.client as mqtt
import time
import random

# --- MQTT 브로커 설정 ---
mqtt_server = "192.168.223.97"
mqtt_port = 1883
mqtt_client_id = "python_chart_publisher"

# Chart 위젯 제어용 토픽
topic_to_chart = "device/status/chart"

# --- 콜백 함수 정의 ---

def on_connect(client, userdata, flags, rc):
    """브로커 연결 성공 시 실행되는 함수입니다."""
    if rc == 0:
        print("✔ MQTT 브로커에 성공적으로 연결되었습니다.")
    else:
        print(f"❌ 연결 실패! 반환 코드(rc): {rc}")

# --- 메인 프로그램 실행 ---
print("--- Chart 위젯 제어 프로그램 ---")
print("3개의 장비 상태 데이터를 1초마다 IoT MQTT Pannel로 보냅니다.")
print("---------------------------------------")

try:
    # MQTT 클라이언트 생성
    client = mqtt.Client(client_id=mqtt_client_id)
    client.on_connect = on_connect

    # 브로커에 연결
    print(f"\nMQTT 브로커 '{mqtt_server}:{mqtt_port}'에 연결 시도 중...")
    client.connect(mqtt_server, mqtt_port, 600)

    # 논블로킹 방식으로 메시지 발행 가능하게 설정
    client.loop_start()

    # 연결이 완료될 때까지 잠시 대기
    while not client.is_connected():
        time.sleep(1)

    # 무한 루프를 돌며 임의의 값 발행
    while True:
        # 3개의 임의의 정수 값 생성 (0-100)
        value1 = random.randint(0, 100)
        value2 = random.randint(0, 100)
        value3 = random.randint(0, 100)

        # 값을 쉼표로 구분된 문자열로 조합
        chart_data = f"{value1},{value2},{value3}"

        # MQTT 메시지 발행
        client.publish(topic_to_chart, chart_data)
        print(f"✔ '{topic_to_chart}' 토픽으로 데이터 '{chart_data}' 발행")

        # 1초 대기
        time.sleep(1)

except KeyboardInterrupt:
    print("\n프로그램을 수동으로 종료합니다.")
except Exception as e:
    print(f"❌ 예외 발생: {e}")

finally:
    # 연결 종료
    if client.is_connected():
        client.loop_stop()
        client.disconnect()
    print("\n연결 종료.")


In [ ]:
#image2
import paho.mqtt.client as mqtt
import time

# --- MQTT 브로커 정보 설정 ---
# Mosquitto가 실행 중인 서버의 IP 주소입니다.
# 같은 컴퓨터에서 실행한다면 'localhost'를 사용하세요.
MQTT_BROKER = "192.168.223.97"
MQTT_PORT = 1883  # 기본 MQTT 포트
MQTT_TOPIC = "iot/image/panel"

# --- 전송할 이미지 URL ---
# 여기에 공개적으로 접근 가능한 이미지 URL을 입력하세요.
IMAGE_URL = "https://picsum.photos/400/300"  # 예시: 임의의 400x300 이미지

# MQTT 브로커에 연결되었을 때 호출되는 콜백 함수
def on_connect(client, userdata, flags, rc):
    """
    브로커와 연결되었을 때 실행되는 함수.
    rc (Return Code): 연결 상태를 나타내는 코드. 0은 성공.
    """
    if rc == 0:
        print("Mosquitto 브로커에 성공적으로 연결되었습니다. 🎉")
        # 연결 성공 후 메시지 발행
        publish_image_url(client, MQTT_TOPIC, IMAGE_URL)
    else:
        print(f"연결 실패, 코드: {rc}")

# MQTT 클라이언트 인스턴스 생성
client = mqtt.Client()

# 콜백 함수 설정
client.on_connect = on_connect

# Mosquitto 브로커에 연결 시도
print(f"Mosquitto 브로커 '{MQTT_BROKER}'에 연결 중...")
try:
    client.connect(MQTT_BROKER, MQTT_PORT, 60)
except Exception as e:
    print(f"연결 오류: {e}")
    exit()

def publish_image_url(client, topic, url):
    """
    특정 토픽으로 이미지 URL을 발행하는 함수.
    """
    print(f"토픽 '{topic}'으로 이미지 URL을 발행합니다.")
    result = client.publish(topic, url)
    status = result[0]
    if status == 0:
        print(f"메시지 발행 성공: '{url}'")
    else:
        print("메시지 발행 실패")

    # 발행 후 루프를 종료
    client.disconnect()

# 네트워크 루프 시작
# 루프는 on_connect 함수가 실행되고 메시지를 발행한 후 종료됩니다.
client.loop_forever()

In [ ]:
import paho.mqtt.client as mqtt
import base64

MQTT_BROKER = "192.168.223.97"
MQTT_TOPIC = "iot/image/base64"
IMAGE_PATH = "c:/jang.png"  # Base64로 보낼 이미지 파일 경로

def on_connect(client, userdata, flags, rc):
    print(f"MQTT 브로커에 연결되었습니다. 결과 코드: {rc}")
    if rc == 0:
        publish_base64_image(client)

def publish_base64_image(client):
    try:
        # 이미지 파일을 바이너리 읽기 모드로 열기
        with open(IMAGE_PATH, "rb") as image_file:
            # 파일을 읽고 Base64로 인코딩
            encoded_string = base64.b64encode(image_file.read()).decode('utf-8')

        print(f"이미지를 Base64로 인코딩하여 발행합니다. 데이터 크기: {len(encoded_string)} Bytes")

        # 메시지 발행
        client.publish(MQTT_TOPIC, encoded_string)

        print("메시지 발행 완료")
        client.disconnect()

    except FileNotFoundError:
        print(f"오류: {IMAGE_PATH} 파일을 찾을 수 없습니다.")
    except Exception as e:
        print(f"발행 오류: {e}")

client = mqtt.Client()
client.on_connect = on_connect

try:
    client.connect(MQTT_BROKER, 1883, 60)
    client.loop_forever()
except Exception as e:
    print(f"연결 오류: {e}")

In [ ]:
#barcode panel
import paho.mqtt.client as mqtt
import serial
import time

# --- MQTT 브로커 설정 ---
mqtt_server = "192.168.223.97"
mqtt_port = 1883
mqtt_client_id = "python_barcode_publisher"

# 바코드 스캐너 데이터 전송용 토픽
topic_to_barcode = "warehouse/scanner/barcode"

# --- 시리얼 포트 설정 ---
# ⚠️⚠️ 사용자의 바코드 스캐너 포트에 맞게 변경해야 합니다. ⚠️⚠️
# Windows: 'COM3', 'COM4' 등
# macOS/Linux: '/dev/tty.usbmodem14101', '/dev/ttyUSB0' 등
# 정확한 포트명은 장치 관리자(Device Manager) 또는 터미널 명령(ls /dev/tty.*)으로 확인하세요.
SERIAL_PORT = "COM3"
BAUD_RATE = 9600  # 스캐너의 전송 속도에 맞춰야 합니다.

# --- 콜백 함수 정의 ---
def on_connect(client, userdata, flags, rc):
    """MQTT 브로커 연결 성공 시 실행되는 함수"""
    if rc == 0:
        print("✔ MQTT 브로커에 성공적으로 연결되었습니다.")
    else:
        print(f"❌ 연결 실패! 반환 코드(rc): {rc}")

# --- 메인 프로그램 실행 ---
print("--- 바코드 스캐너 연동 프로그램 ---")
print(f"스캐너('{SERIAL_PORT}')에서 읽은 데이터를 MQTT로 전송합니다.")
print("---------------------------------------")

try:
    # 시리얼 포트 연결
    ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
    print(f"✔ 시리얼 포트 '{SERIAL_PORT}'에 연결되었습니다.")
    time.sleep(2)  # 스캐너 초기화 시간 대기

    # MQTT 클라이언트 생성
    client = mqtt.Client(client_id=mqtt_client_id)
    client.on_connect = on_connect

    # 브로커에 연결
    print(f"\nMQTT 브로커 '{mqtt_server}:{mqtt_port}'에 연결 시도 중...")
    client.connect(mqtt_server, mqtt_port, 60)
    client.loop_start()

    # 연결이 완료될 때까지 잠시 대기
    while not client.is_connected():
        time.sleep(1)

    # 무한 루프를 돌며 시리얼 포트 데이터 읽기
    while True:
        if ser.in_waiting > 0:
            # 시리얼 포트에서 한 줄 읽기 (바코드 데이터)
            barcode_data_bytes = ser.readline()

            # 읽은 데이터를 UTF-8로 디코딩하고 공백 제거
            barcode_data = barcode_data_bytes.decode('utf-8').strip()

            if barcode_data:
                # MQTT 메시지 발행
                client.publish(topic_to_barcode, barcode_data)
                print(f"✔ '{topic_to_barcode}' 토픽으로 바코드 데이터 '{barcode_data}' 발행")

        # CPU 점유율을 낮추기 위해 잠시 대기
        time.sleep(0.1)

except serial.SerialException as se:
    print(f"❌ 시리얼 포트 오류: 포트 '{SERIAL_PORT}'를 열 수 없습니다. 포트가 올바른지, 사용 중인 포트가 아닌지 확인하세요.")
except KeyboardInterrupt:
    print("\n프로그램을 수동으로 종료합니다.")
except Exception as e:
    print(f"❌ 예외 발생: {e}")

finally:
    # 연결 종료
    if 'client' in locals() and client.is_connected():
        client.loop_stop()
        client.disconnect()
        print("\nMQTT 연결 종료.")
    if 'ser' in locals() and ser.is_open:
        ser.close()
        print("시리얼 포트 종료.")

In [ ]:
#

import paho.mqtt.client as mqtt
import time

# --- MQTT 브로커 정보 설정 ---
# 여러분의 MQTT 브로커 IP 주소 또는 도메인으로 변경하세요.
MQTT_BROKER = "192.168.219.100"
MQTT_PORT = 1883
MQTT_CLIENT_ID = "python_url_launcher"

# URL Launcher 패널을 제어할 토픽
# 이 토픽은 IoT MQTT Panel 앱에서 설정할 토픽과 동일해야 합니다.
TOPIC_TO_LAUNCH = "control/open_website"

# --- 콜백 함수 정의 ---
def on_connect(client, userdata, flags, rc):
    """브로커 연결 성공 시 실행되는 함수입니다."""
    if rc == 0:
        print("✔ MQTT 브로커에 성공적으로 연결되었습니다.")
        # 연결 성공 후 메시지를 발행하여 URL 실행을 트리거합니다.
        publish_message(client)
    else:
        print(f"❌ 연결 실패! 반환 코드(rc): {rc}")

def publish_message(client):
    """
    URL 실행을 위한 MQTT 메시지를 발행하는 함수입니다.
    페이로드(메시지 내용)는 중요하지 않습니다.
    """
    print(f"\n'{TOPIC_TO_LAUNCH}' 토픽으로 메시지를 발행합니다.")

    # URL 실행을 트리거하는 메시지 발행
    result = client.publish(TOPIC_TO_LAUNCH, payload="", qos=0, retain=False)

    status = result[0]
    if status == 0:
        print("✔ 메시지 발행 성공! 앱에서 URL이 실행될 것입니다.")
    else:
        print(f"❌ 메시지 발행 실패! 반환 코드: {status}")

    # 메시지 발행 후 연결 종료
    client.disconnect()

# --- 메인 프로그램 실행 ---
print("--- URL Launcher 제어 프로그램 ---")
print("MQTT 메시지 발행 시, IoT MQTT Panel에서 URL을 실행합니다.")
print("---------------------------------------")

try:
    # MQTT 클라이언트 생성
    client = mqtt.Client(client_id=MQTT_CLIENT_ID)
    client.on_connect = on_connect

    # 브로커에 연결
    print(f"\nMQTT 브로커 '{MQTT_BROKER}:{MQTT_PORT}'에 연결 시도 중...")
    client.connect(MQTT_BROKER, MQTT_PORT, 60)

    # 네트워크 루프 시작 (연결 및 메시지 발행을 기다림)
    client.loop_forever()

except KeyboardInterrupt:
    print("\n프로그램을 수동으로 종료합니다.")
except Exception as e:
    print(f"❌ 예외 발생: {e}")

finally:
    # 프로그램 종료 시 연결 정리
    if 'client' in locals() and client.is_connected():
        client.disconnect()
    print("\n프로그램 종료.")


In [ ]:
import paho.mqtt.client as mqtt
import time
import random
import base64

# --- MQTT 브로커 설정 ---
mqtt_server = "192.168.219.100"  # 여러분의 브로커 IP로 변경하세요.
mqtt_port = 1883
mqtt_client_id = "iot_panel_integrator"

# --- 각 위젯에 대한 토픽 설정 ---
topic_url_launcher = "control/open_website"  # URL Launcher용
topic_chart = "device/status/chart"  # 차트용
topic_image = "camera/image"  # 이미지용
topic_text = "sensor/temperature"  # 텍스트용

# --- 이미지 파일 및 URL 설정 ---
# Base64로 보낼 이미지 파일 (같은 폴더에 있어야 합니다.)
IMAGE_FILE_PATH = "c:/jang.png"
# URL로 보낼 이미지
IMAGE_URL = "https://picsum.photos/400/300"

# --- 콜백 함수 정의 ---
def on_connect(client, userdata, flags, rc):
    """브로커 연결 성공 시 실행되는 함수입니다."""
    if rc == 0:
        print("✔ MQTT 브로커에 성공적으로 연결되었습니다.")
    else:
        print(f"❌ 연결 실패! 반환 코드(rc): {rc}")

def send_data(client):
    """
    각 위젯으로 데이터를 보내는 함수
    """
    # 1. URL Launcher 활성화
    client.publish(topic_url_launcher, payload="")
    print(f"✔ URL Launcher 메시지 발행: '{topic_url_launcher}'")

    # 2. 차트 데이터 전송
    chart_value1 = random.randint(0, 100)
    chart_value2 = random.randint(0, 100)
    chart_value3 = random.randint(0, 100)
    chart_data = f"{chart_value1},{chart_value2},{chart_value3}"
    client.publish(topic_chart, chart_data)
    print(f"✔ 차트 데이터 발행: '{topic_chart}' -> {chart_data}")

    # 3. 이미지 데이터 전송 (번갈아가며)
    global image_mode
    if image_mode == "url":
        client.publish(topic_image, IMAGE_URL)
        print(f"✔ 이미지 URL 발행: '{topic_image}' -> {IMAGE_URL}")
        image_mode = "base64"
    else:
        try:
            with open(IMAGE_FILE_PATH, "rb") as image_file:
                encoded_string = base64.b64encode(image_file.read()).decode('utf-8')
            client.publish(topic_image, f"base64_payload:{encoded_string}")
            print(f"✔ 이미지 Base64 발행: '{topic_image}' -> (Base64 data)")
            image_mode = "url"
        except FileNotFoundError:
            print(f"❌ '{IMAGE_FILE_PATH}' 파일을 찾을 수 없습니다. Base64 전송 건너뜀.")

    # 4. 텍스트 데이터 전송
    current_temp = round(random.uniform(20.0, 30.0), 1)
    client.publish(topic_text, f"현재 온도: {current_temp}°C")
    print(f"✔ 텍스트 데이터 발행: '{topic_text}' -> '현재 온도: {current_temp}°C'")

# --- 메인 프로그램 실행 ---
print("--- IoT MQTT Panel 통합 제어 프로그램 ---")
print(f"MQTT 브로커 '{mqtt_server}:{mqtt_port}'에 연결 시도 중...")
print("---------------------------------------")

client = mqtt.Client(client_id=mqtt_client_id)
client.on_connect = on_connect
image_mode = "url"

try:
    client.connect(mqtt_server, mqtt_port, 600)
    client.loop_start()

    while not client.is_connected():
        time.sleep(1)

    while True:
        send_data(client)
        print("---------------------------------------")
        time.sleep(5)  # 5초마다 데이터 전송

except KeyboardInterrupt:
    print("\n프로그램을 수동으로 종료합니다.")
except Exception as e:
    print(f"❌ 예외 발생: {e}")
finally:
    if client.is_connected():
        client.loop_stop()
        client.disconnect()
    print("\n연결 종료.")

In [ ]:
#mqtt 와 blue bluetooth통신
'''블루투스 데이터를 MQTT로 변환하는 원리
직접 연결은 불가능하지만, 중간 매개체(Gateway 또는 Bridge)를 활용하여 두 프로토콜을 연결하는 방식입니다.
블루투스 기기: 블루투스 센서(예: BLE 온도 센서)가 데이터를 전송합니다.
블루투스-MQTT 브릿지: 라즈베리 파이, ESP32, 또는 PC와 같은 장치가 블루투스 통신을 수신합니다. 이 장치는 수신한 데이터를 MQTT 메시지로 변환하여 MQTT 브로커에 발행합니다.
MQTT 브로커: 브릿지로부터 MQTT 메시지를 받아서 관리합니다.
IoT MQTT Panel: 앱이 MQTT 브로커에 연결하여 필요한 토픽을 구독하고, 수신된 데이터를 화면에 표시합니다.
이 방식은 블루투스 기기와 MQTT 생태계를 완벽하게 통합하여, IoT MQTT Panel에서 블루투스 기기의 데이터를 시각화하고 제어할 수 있게 해줍니다.'''
import asyncio
import bleak
import paho.mqtt.client as mqtt
import time

# --- MQTT 브로커 설정 ---
mqtt_server = "192.168.223.97"  # 여러분의 브로커 IP
mqtt_port = 1883
mqtt_client_id = "ble_mqtt_bridge"

# --- BLE 장치 설정 ---
# ⚠️⚠️ 여기에 실제 BLE 장치의 MAC 주소와 서비스, 특성(characteristic) UUID를 입력하세요. ⚠️⚠️
BLE_DEVICE_MAC = "AA:BB:CC:DD:EE:FF"  # 예시 MAC 주소
TEMP_CHAR_UUID = "00002A1C-0000-1000-8000-00805F9B34FB"  # 예시: 온도 특성 UUID

# --- 토픽 설정 ---
mqtt_topic = "ble/sensor/temperature"

# --- MQTT 클라이언트 생성 ---
mqtt_client = mqtt.Client(client_id=mqtt_client_id)

def on_connect(client, userdata, flags, rc):
    """MQTT 연결 성공 시 호출되는 함수"""
    if rc == 0:
        print("✔ MQTT 브로커 연결 성공!")
    else:
        print(f"❌ MQTT 연결 실패: {rc}")

mqtt_client.on_connect = on_connect
mqtt_client.connect(mqtt_server, mqtt_port, 60)
mqtt_client.loop_start()

# --- BLE 데이터 처리 및 MQTT 발행 ---
async def run_ble_to_mqtt_bridge(mac_address, char_uuid):
    print(f"\nBLE 장치 '{mac_address}'에 연결 시도 중...")
    try:
        async with bleak.BleakClient(mac_address) as client:
            print(f"✔ BLE 장치에 연결되었습니다: {client.is_connected}")

            # GATT 특성(Characteristic) 구독
            def notification_handler(sender, data):
                """알림을 받을 때마다 호출되는 함수"""
                # 받은 데이터를 디코딩 (예: 바이트를 문자열로)
                temperature = float(data.decode('utf-8'))

                # MQTT 메시지 발행
                mqtt_client.publish(mqtt_topic, temperature)
                print(f"✔ 데이터 발행: '{mqtt_topic}' -> {temperature}°C")

            await client.start_notify(char_uuid, notification_handler)

            # 연결을 유지하며 알림을 기다립니다.
            print("데이터 수신 대기 중...")
            while True:
                await asyncio.sleep(1)

    except Exception as e:
        print(f"❌ 연결 또는 통신 오류: {e}")
    finally:
        mqtt_client.loop_stop()
        mqtt_client.disconnect()
        print("프로그램 종료.")

if __name__ == "__main__":
    asyncio.run(run_ble_to_mqtt_bridge(BLE_DEVICE_MAC, TEMP_CHAR_UUID))

In [ ]:
# 블루투스 송수신 가능 프로그램
import asyncio
import bleak
import paho.mqtt.client as mqtt
import time

# --- MQTT 브로커 설정 ---
mqtt_server = "192.168.223.97"  # 여러분의 브로커 IP
mqtt_port = 1883
mqtt_client_id = "ble_mqtt_two_way_bridge"

# --- BLE 장치 설정 ---
BLE_DEVICE_MAC = "AA:BB:CC:DD:EE:FF"  # ⚠️ 실제 BLE 장치 MAC 주소
TEMP_CHAR_UUID = "00002A1C-0000-1000-8000-00805F9B34FB"  # ⚠️ BLE 장치에서 읽을 특성 UUID
CONTROL_CHAR_UUID = "0000ABCD-0000-1000-8000-00805F9B34FB"  # ⚠️ BLE 장치에 쓸 특성 UUID

# --- 토픽 설정 ---
mqtt_topic_from_ble = "ble/sensor/temperature"  # BLE->MQTT (데이터 수신)
mqtt_topic_to_ble = "ble/control/command"        # MQTT->BLE (명령 송신)

# --- 전역 변수 ---
ble_client = None

# --- MQTT 콜백 함수 정의 ---
def on_connect(client, userdata, flags, rc):
    """MQTT 연결 성공 시 호출되는 함수"""
    if rc == 0:
        print("✔ MQTT 브로커 연결 성공!")
        client.subscribe(mqtt_topic_to_ble)
    else:
        print(f"❌ MQTT 연결 실패: {rc}")

def on_message(client, userdata, msg):
    """MQTT 메시지 수신 시 호출되는 함수 (MQTT->BLE)"""
    command = msg.payload.decode('utf-8')
    print(f"\n✔ MQTT 메시지 수신: '{msg.topic}' -> '{command}'")

    # 받은 명령을 BLE 장치로 전송
    asyncio.run(send_to_ble(command))

# --- BLE 통신 함수 정의 ---
async def send_to_ble(command):
    """BLE 장치로 데이터를 쓰는 함수 (BLE 송신)"""
    global ble_client
    if ble_client and ble_client.is_connected:
        try:
            # 문자열 명령어를 바이트로 변환하여 전송
            await ble_client.write_gatt_char(CONTROL_CHAR_UUID, command.encode('utf-8'))
            print(f"✔ BLE 장치에 명령 전송: '{command}'")
        except Exception as e:
            print(f"❌ BLE 쓰기 오류: {e}")
    else:
        print("❌ BLE 장치에 연결되지 않았습니다.")

async def run_ble_to_mqtt_bridge():
    """BLE 장치와 MQTT 브로커를 연결하는 메인 함수"""
    global ble_client
    print(f"\nBLE 장치 '{BLE_DEVICE_MAC}'에 연결 시도 중...")

    try:
        ble_client = bleak.BleakClient(BLE_DEVICE_MAC)
        await ble_client.connect()
        print(f"✔ BLE 장치에 연결되었습니다: {ble_client.is_connected}")

        # BLE 데이터 수신 (BLE -> MQTT)
        def ble_notification_handler(sender, data):
            """BLE 알림 수신 시 호출되는 핸들러"""
            try:
                # 받은 바이트 데이터를 float로 변환하여 MQTT로 발행
                temperature = float(data.decode('utf-8'))
                mqtt_client.publish(mqtt_topic_from_ble, temperature)
                print(f"✔ 데이터 발행: '{mqtt_topic_from_ble}' -> {temperature}°C")
            except Exception as e:
                print(f"❌ 데이터 처리 오류: {e}")

        await ble_client.start_notify(TEMP_CHAR_UUID, ble_notification_handler)

        # 프로그램 실행 상태 유지
        print("양방향 통신 대기 중...")
        await asyncio.Event().wait()

    except Exception as e:
        print(f"❌ 연결 또는 통신 오류: {e}")
    finally:
        if ble_client and ble_client.is_connected:
            await ble_client.disconnect()
        print("BLE 연결 종료.")

# --- 메인 실행 ---
if __name__ == "__main__":
    # MQTT 클라이언트 생성 및 연결
    mqtt_client = mqtt.Client(client_id=mqtt_client_id)
    mqtt_client.on_connect = on_connect
    mqtt_client.on_message = on_message

    try:
        mqtt_client.connect(mqtt_server, mqtt_port, 60)
        mqtt_client.loop_start()

        # MQTT 연결이 완료될 때까지 대기
        while not mqtt_client.is_connected():
            time.sleep(1)

        # 비동기 루프 시작
        asyncio.run(run_ble_to_mqtt_bridge())

    except KeyboardInterrupt:
        print("\n프로그램을 수동으로 종료합니다.")
    except Exception as e:
        print(f"❌ 예외 발생: {e}")
    finally:
        if mqtt_client.is_connected():
            mqtt_client.loop_stop()
            mqtt_client.disconnect()
            print("MQTT 연결 종료.")

In [ ]:
#직렬통신
import serial
import paho.mqtt.client as mqtt
import time

# --- MQTT 브로커 설정 ---
mqtt_server = "192.168.223.97"
mqtt_port = 1883
mqtt_client_id = "basys3_two_way_bridge"

# --- 시리얼 통신 설정 ---
SERIAL_PORT = "COM28"  # ⚠️ Basys3의 시리얼 포트명으로 변경하세요. ⚠️
BAUD_RATE = 9600

# --- MQTT 토픽 설정 ---
# 1. Basys3에서 보낸 데이터를 수신할 토픽 (송신)
basys3_to_app_topic = "basys3/data"
# 2. 앱에서 Basys3로 보낼 명령을 수신할 토픽 (수신)
app_to_basys3_topic = "basys3/command"

# --- 콜백 함수 정의 ---
def on_connect(client, userdata, flags, rc):
    """MQTT 브로커 연결 성공 시 호출되는 함수"""
    if rc == 0:
        print("✔ MQTT 브로커 연결 성공!")
        # 연결 성공 후 메시지를 수신할 토픽을 구독
        client.subscribe(app_to_basys3_topic)
    else:
        print(f"❌ MQTT 연결 실패: {rc}")

def on_message(client, userdata, msg):
    """구독한 토픽으로 메시지가 도착했을 때 호출되는 함수"""
    try:
        # 받은 메시지 디코딩
        command = msg.payload.decode('utf-8')
        print(f"\n✔ MQTT 메시지 수신: '{msg.topic}' -> '{command}'")

        # Basys3로 직렬 통신 데이터 전송
        # Basys3는 'LED_ON' 또는 'LED_OFF' 명령을 인식하도록 FPGA 코드가 구현되어야 함.
        if ser.is_open:
            ser.write(command.encode('utf-8'))
            print(f"✔ Basys3로 직렬 데이터 전송: '{command}'")
    except Exception as e:
        print(f"❌ 메시지 처리 오류: {e}")

# --- 메인 프로그램 실행 ---
print("--- Basys3 양방향 통신 브릿지 프로그램 ---")

try:
    # 시리얼 포트 연결
    ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
    print(f"✔ 시리얼 포트 '{SERIAL_PORT}'에 연결되었습니다.")
    time.sleep(2)

    # MQTT 클라이언트 생성 및 콜백 함수 설정
    mqtt_client = mqtt.Client(client_id=mqtt_client_id)
    mqtt_client.on_connect = on_connect
    mqtt_client.on_message = on_message

    # MQTT 브로커 연결
    mqtt_client.connect(mqtt_server, mqtt_port, 60)
    mqtt_client.loop_start()

    while not mqtt_client.is_connected():
        time.sleep(1)

    # 무한 루프를 돌며 Basys3에서 데이터 읽고 MQTT로 발행
    while True:
        if ser.in_waiting > 0:
            line_bytes = ser.readline()
            data = line_bytes.decode('utf-8').strip()

            if data:
                # Basys3에서 받은 데이터를 MQTT로 발행 (송신)
                mqtt_client.publish(basys3_to_app_topic, data)
                print(f"✔ 데이터 발행: '{basys3_to_app_topic}' -> '{data}'")

        time.sleep(0.1)

except serial.SerialException as se:
    print(f"❌ 시리얼 포트 오류: 포트 '{SERIAL_PORT}'를 열 수 없습니다.")
except KeyboardInterrupt:
    print("\n프로그램을 수동으로 종료합니다.")
except Exception as e:
    print(f"❌ 예외 발생: {e}")
finally:
    if 'mqtt_client' in locals() and mqtt_client.is_connected():
        mqtt_client.loop_stop()
        mqtt_client.disconnect()
        print("\nMQTT 연결 종료.")
    if 'ser' in locals() and ser.is_open:
        ser.close()
        print("시리얼 포트 종료.")

In [ ]:
import serial
import paho.mqtt.client as mqtt
import time

# --- MQTT 브로커 설정 ---
mqtt_server = "192.168.223.97"
mqtt_port = 1883
mqtt_client_id = "arduino_led_control"

# --- 시리얼 통신 설정 ---
# ⚠️ 아두이노의 시리얼 포트명을 확인하여 정확하게 변경하세요. (예: 'COM3' 또는 '/dev/ttyACM0')
SERIAL_PORT = "COM28"
BAUD_RATE = 9600

# --- MQTT 토픽 설정 ---
# 핸드폰 -> 아두이노 (명령 송신)
app_to_arduino_topic = "control/led"
# 아두이노 -> 핸드폰 (상태 피드백)
arduino_to_app_topic = "status/led"

# --- 전역 변수 ---
ser = None
mqtt_client = None

# --- MQTT 콜백 함수 정의 ---
def on_connect(client, userdata, flags, rc):
    """MQTT 브로커 연결 성공 시 호출되는 함수"""
    if rc == 0:
        print("✔ MQTT 브로커 연결 성공!")
        # 연결 성공 후 제어 메시지를 수신할 토픽을 구독
        client.subscribe(app_to_arduino_topic)
    else:
        print(f"❌ MQTT 연결 실패: {rc}")

def on_message(client, userdata, msg):
    """구독한 토픽으로 메시지가 도착했을 때 호출되는 함수"""
    try:
        # 받은 메시지 디코딩
        command = msg.payload.decode('utf-8')
        print(f"\n✔ MQTT 메시지 수신: '{msg.topic}' -> '{command}'")

        if ser and ser.is_open:
            # 아두이노가 한 줄의 끝을 인식하도록 개행 문자 '\n'을 추가
            ser.write((command + '\n').encode('utf-8'))
            print(f"✔ 아두이노로 시리얼 데이터 전송: '{command}'")
        else:
            print("❌ 시리얼 포트가 열려 있지 않습니다.")

    except Exception as e:
        print(f"❌ 메시지 처리 오류: {e}")

# --- 메인 프로그램 실행 ---
print("--- 핸드폰-아두이노 양방향 통신 브릿지 프로그램 ---")

try:
    # 시리얼 포트 연결
    ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
    print(f"✔ 시리얼 포트 '{SERIAL_PORT}'에 연결되었습니다.")
    time.sleep(2)

    # 시리얼 통신 버퍼를 비워 잔여 데이터로 인한 오류 방지
    ser.flushInput()
    ser.flushOutput()

    # MQTT 클라이언트 생성 및 콜백 함수 설정
    mqtt_client = mqtt.Client(client_id=mqtt_client_id)
    mqtt_client.on_connect = on_connect
    mqtt_client.on_message = on_message

    # MQTT 브로커 연결
    mqtt_client.connect(mqtt_server, mqtt_port, 60)
    mqtt_client.loop_start()

    # MQTT 연결이 완료될 때까지 대기
    while not mqtt_client.is_connected():
        time.sleep(1)

    # 무한 루프를 돌며 아두이노에서 데이터 읽고 MQTT로 발행
    while True:
        if ser and ser.in_waiting > 0:
            line_bytes = ser.readline()
            data = line_bytes.decode('utf-8').strip()

            if data:
                # 아두이노에서 받은 상태를 MQTT로 발행 (핸드폰으로 피드백)
                mqtt_client.publish(arduino_to_app_topic, data)
                print(f"✔ 상태 피드백 발행: '{arduino_to_app_topic}' -> '{data}'")

        time.sleep(0.1)

except serial.SerialException as se:
    print(f"❌ 시리얼 포트 오류: 포트 '{SERIAL_PORT}'를 열 수 없습니다. 포트 번호를 확인하거나, 다른 프로그램이 사용 중인지 확인하세요.")
except KeyboardInterrupt:
    print("\n프로그램을 수동으로 종료합니다.")
except Exception as e:
    print(f"❌ 예외 발생: {e}")
finally:
    if 'mqtt_client' in locals() and mqtt_client and mqtt_client.is_connected():
        mqtt_client.loop_stop()
        mqtt_client.disconnect()
        print("\nMQTT 연결 종료.")
    if 'ser' in locals() and ser and ser.is_open:
        ser.close()
        print("시리얼 포트 종료.")

In [ ]:
#ydlidar x2  ex
import struct
import csv
import matplotlib.pyplot as plt
import numpy as np

def parse_lidar_packet(packet_bytes):
    if len(packet_bytes) < 10:
        raise ValueError("패킷 길이가 너무 짧습니다.")

    # PH: Packet Header (2B)
    ph = struct.unpack_from('<H', packet_bytes, 0)[0]
    # NOTE: example_packet에 맞게 헤더 값을 0xAA55로 수정하거나,
    # 예시 패킷을 0x5aA5로 수정해야 합니다. 여기서는 패킷을 수정했습니다.
    if ph != 0x5aA5:
        raise ValueError("잘못된 패킷 헤더입니다.")


    ct = packet_bytes[2]
    is_start_of_lap = ct & 0x01
    packet_type = 'Lap Start' if is_start_of_lap else 'Point Cloud'
    lsn = packet_bytes[3]
    fsa = struct.unpack_from('<H', packet_bytes, 4)[0] / 100.0
    lsa = struct.unpack_from('<H', packet_bytes, 6)[0] / 100.0
    cs = struct.unpack_from('<H', packet_bytes, 8)[0]
    samples = []
    offset = 10
    for i in range(lsn):
        if offset + 2 > len(packet_bytes):
            break
        si_raw = struct.unpack_from('<H', packet_bytes, offset)[0]
        distance = si_raw & 0x3FFF
        interference_flag = (si_raw >> 14) & 0x03
        samples.append({
            'index': i,
            'distance_mm': distance,
            'interference': interference_flag
        })
        offset += 2
    return {
        'packet_type': packet_type,
        'sample_count': lsn,
        'start_angle_deg': fsa,
        'end_angle_deg': lsa,
        'checksum': cs,
        'samples': samples
    }

def save_samples_to_csv(samples, filename='lidar_samples.csv'):
    with open(filename, mode='w', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=['index', 'distance_mm', 'interference'])
        writer.writeheader()
        for sample in samples:
            writer.writerow(sample)
    print(f"CSV 파일로 저장 완료: {filename}")

def visualize_lidar_data(samples, start_angle, end_angle):
    count = len(samples)
    angles = np.linspace(start_angle, end_angle, count)
    distances = [s['distance_mm'] for s in samples]

    angles_rad = np.deg2rad(angles)
    plt.figure(figsize=(6,6))
    ax = plt.subplot(111, polar=True)
    ax.plot(angles_rad, distances, marker='o', linestyle='-', color='blue')
    ax.set_title("LiDAR 스캔 시각화", fontsize=14)
    plt.show()
# 예시: 바이너리 패킷 데이터
# 헤더를 0x5aA5 (\x5a\xa5)로 수정
example_packet = b'\xa5\x5a\x00\x02\x10\x27\x90\x27\xab\xcd\x34\x12\x78\x56'
parsed = parse_lidar_packet(example_packet)
save_samples_to_csv(parsed['samples'])
visualize_lidar_data(parsed['samples'], parsed['start_angle_deg'], parsed['end_angle_deg'])


try:
    parsed = parse_lidar_packet(example_packet)
    print(parsed)
    save_samples_to_csv(parsed['samples'])
    visualize_lidar_data(parsed['samples'], parsed['start_angle_deg'], parsed['end_angle_deg'])
except ValueError as e:
    print(f"오류 발생: {e}")


In [ ]:
def filter_by_distance(samples, min_mm=100, max_mm=3000):
    return [s for s in samples if min_mm <= s['distance_mm'] <= max_mm]

def polar_to_cartesian(samples):
    points = []
    for sample in samples:
        angle_deg = sample['index'] * 360 / len(samples)
        angle_rad = np.deg2rad(angle_deg)
        r = sample['distance_mm']
        x = r * np.cos(angle_rad)
        y = r * np.sin(angle_rad)
        points.append((x, y))
    return points
def visualize_2d_map(points):
    x_vals, y_vals = zip(*points)
    plt.figure(figsize=(8, 8))
    plt.scatter(x_vals, y_vals, s=5, c='blue')
    plt.title("LiDAR 2D 맵 (거리 필터링 적용)", fontsize=14)
    plt.xlabel("X (mm)")
    plt.ylabel("Y (mm)")
    plt.axis('equal')
    plt.grid(True)
    plt.show()
 def visualize_2d_map(points):
    x_vals, y_vals = zip(*points)
    plt.figure(figsize=(8, 8))
    plt.scatter(x_vals, y_vals, s=5, c='blue')
    plt.title("LiDAR 2D 맵 (거리 필터링 적용)", fontsize=14)
    plt.xlabel("X (mm)")
    plt.ylabel("Y (mm)")
    plt.axis('equal')
    plt.grid(True)
    plt.show()